In [33]:
import pandas as pd
import numpy as np

In [34]:
df = pd.read_csv("../data/raw/results.csv")

In [35]:
df["date"] = pd.to_datetime(df["date"])

In [36]:
completed_matches = df[
    df["home_score"].notna() &
    df["away_score"].notna()
].copy()

completed_matches = completed_matches.sort_values("date")

In [37]:
future_fixtures = df[
    df["home_score"].isna() |
    df["away_score"].isna()
].copy()

In [38]:
def get_result(row):
    if row["home_score"] > row["away_score"]:
        return "H"
    elif row["home_score"] < row["away_score"]:
        return "A"
    return "D"

completed_matches["result"] = completed_matches.apply(get_result, axis=1)

In [39]:
completed_matches.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False,D
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False,H
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False,H
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False,D
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False,H


In [40]:
def get_team_matches(team, date, matches):

    previous_matches = matches[
        (
            (matches["home_team"] == team) |
            (matches["away_team"] == team)
        )
        &
        (matches["date"] < date)
    ]

    return previous_matches.sort_values("date", ascending = False)

In [41]:
get_team_matches("Argentina", pd.Timestamp("2026-01-01"), completed_matches).head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result
48893,2025-11-14,Angola,Argentina,0.0,2.0,Friendly,Luanda,Angola,False,A
48816,2025-10-14,Puerto Rico,Argentina,0.0,6.0,Friendly,Fort Lauderdale,United States,True,A
48730,2025-10-10,Argentina,Venezuela,1.0,0.0,Friendly,Miami Gardens,United States,True,H
48646,2025-09-09,Ecuador,Argentina,1.0,0.0,FIFA World Cup qualification,Guayaquil,Ecuador,False,H
48533,2025-09-04,Argentina,Venezuela,3.0,0.0,FIFA World Cup qualification,Buenos Aires,Argentina,False,H


In [42]:
def get_recent_matches(team, date, matches, n=10):
    previous_matches = get_team_matches(team, date, matches)
    return previous_matches.head(n)

In [43]:
argentina_recent = get_recent_matches("Argentina", pd.Timestamp("2026-01-01"), completed_matches)

argentina_recent.head(10)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,result
48893,2025-11-14,Angola,Argentina,0.0,2.0,Friendly,Luanda,Angola,False,A
48816,2025-10-14,Puerto Rico,Argentina,0.0,6.0,Friendly,Fort Lauderdale,United States,True,A
48730,2025-10-10,Argentina,Venezuela,1.0,0.0,Friendly,Miami Gardens,United States,True,H
48646,2025-09-09,Ecuador,Argentina,1.0,0.0,FIFA World Cup qualification,Guayaquil,Ecuador,False,H
48533,2025-09-04,Argentina,Venezuela,3.0,0.0,FIFA World Cup qualification,Buenos Aires,Argentina,False,H
48438,2025-06-10,Argentina,Colombia,1.0,1.0,FIFA World Cup qualification,Buenos Aires,Argentina,False,D
48329,2025-06-05,Chile,Argentina,0.0,1.0,FIFA World Cup qualification,Santiago,Chile,False,A
48260,2025-03-25,Argentina,Brazil,4.0,1.0,FIFA World Cup qualification,Buenos Aires,Argentina,False,H
48170,2025-03-21,Uruguay,Argentina,0.0,1.0,FIFA World Cup qualification,Montevideo,Uruguay,False,A
48017,2024-11-19,Argentina,Peru,1.0,0.0,FIFA World Cup qualification,Buenos Aires,Argentina,False,H


In [44]:
def calculate_win_rate(team, date, matches, n=10):
    recent_matches = get_recent_matches(team, date, matches, n)

    if len(recent_matches) == 0:
        return np.nan
    
    wins = 0

    for _, match in recent_matches.iterrows():

        if(
            match["home_team"] == team 
            and match["home_score"] > match["away_score"]
        ):
            wins += 1
        
        elif(
            match["away_team"] == team 
            and match["away_score"] > match["home_score"]
        ):
            wins += 1
    
    return wins/len(recent_matches)

In [45]:
print("Argentina:", calculate_win_rate("Argentina", pd.Timestamp("2026-01-01"), completed_matches))

print("Brazil:", calculate_win_rate("Brazil", pd.Timestamp("2026-01-01"), completed_matches))

Argentina: 0.8
Brazil: 0.5


In [46]:
def calculate_avg_goals_scored(team, date, matches, n=10):
    recent_matches = get_recent_matches(team, date, matches, n)

    if(len(recent_matches) == 0):
        return np.nan

    goals = []

    for _, match in recent_matches.iterrows():
        if match["home_team"] == team:
            goals.append(match["home_score"])
        
        elif match["away_team"] == team:
            goals.append(match["away_score"])
    
    return np.mean(goals)

In [47]:
print("Argentina:", calculate_avg_goals_scored("Argentina", pd.Timestamp("2026-01-01"), completed_matches))
print("Brazil:", calculate_avg_goals_scored("Brazil", pd.Timestamp("2026-01-01"), completed_matches))

Argentina: 2.0
Brazil: 1.7


In [48]:
def calculate_avg_goals_conceded(team, date, matches, n=10):
    recent_matches = get_recent_matches(team, date, matches, n)

    if(len(recent_matches) == 0):
        return np.nan

    goals = []

    for _, match in recent_matches.iterrows():
        if match["home_team"] == team:
            goals.append(match["away_score"])
        
        elif match["away_team"] == team:
            goals.append(match["home_score"])
    
    return np.mean(goals)

In [49]:
print("Argentina:", calculate_avg_goals_conceded("Argentina", pd.Timestamp("2026-01-01"), completed_matches))
print("Brazil:", calculate_avg_goals_conceded("Brazil", pd.Timestamp("2026-01-01"), completed_matches))

Argentina: 0.3
Brazil: 1.0


In [50]:
def calculate_goal_difference(team, date, matches, n = 10):
    scored_goals = calculate_avg_goals_scored(team, date, matches, n)
    conceded_goals = calculate_avg_goals_conceded(team, date, matches, n)

    return scored_goals - conceded_goals

In [51]:
print("Argentina:", calculate_goal_difference("Argentina", pd.Timestamp("2026-01-01"), completed_matches))
print("Brazil:", calculate_goal_difference("Brazil", pd.Timestamp("2026-01-01"), completed_matches))

Argentina: 1.7
Brazil: 0.7


In [52]:
print("Argentina: ")
print("Win Rate:", calculate_win_rate("Argentina", pd.Timestamp("2026-01-01"), completed_matches))
print("Goals Scored:", calculate_avg_goals_scored("Argentina", pd.Timestamp("2026-01-01"), completed_matches))
print("Goals Conceded:", calculate_avg_goals_conceded("Argentina", pd.Timestamp("2026-01-01"), completed_matches))
print("Goal Difference:", calculate_goal_difference("Argentina", pd.Timestamp("2026-01-01"), completed_matches))

print("Brazil: ")
print("Win Rate:", calculate_win_rate("Brazil", pd.Timestamp("2026-01-01"), completed_matches))
print("Goals Scored:", calculate_avg_goals_scored("Brazil", pd.Timestamp("2026-01-01"), completed_matches))
print("Goals Conceded:", calculate_avg_goals_conceded("Brazil", pd.Timestamp("2026-01-01"), completed_matches))
print("Goal Difference:", calculate_goal_difference("Brazil", pd.Timestamp("2026-01-01"), completed_matches))

Argentina: 
Win Rate: 0.8
Goals Scored: 2.0
Goals Conceded: 0.3
Goal Difference: 1.7
Brazil: 
Win Rate: 0.5
Goals Scored: 1.7
Goals Conceded: 1.0
Goal Difference: 0.7


In [53]:
sample_matches = completed_matches[completed_matches["date"] >= '2024-01-01'].copy()
sample_matches.shape

(2540, 10)

In [54]:
def create_match_features(match, matches):
    date = match["date"]

    home_team = match["home_team"]
    away_team = match["away_team"]

    return{
        "date": date,
        "home_team": home_team,
        "away_team": away_team,

        "home_win_rate": calculate_win_rate(home_team, date, matches),
        "away_win_rate": calculate_win_rate(away_team, date, matches),

        "home_avg_goals_scored": calculate_avg_goals_scored(home_team, date, matches),
        "away_avg_goals_scored": calculate_avg_goals_scored(away_team, date, matches),

        "home_avg_goals_conceded": calculate_avg_goals_conceded(home_team, date, matches),
        "away_avg_goals_conceded": calculate_avg_goals_conceded(away_team, date, matches),

        "home_goal_diff": calculate_goal_difference(home_team, date, matches),
        "away_goal_diff": calculate_goal_difference(away_team, date, matches),

        "neutral": int(match["neutral"]),
        "result": match["result"]
    }

In [55]:
#Test 
sample_match = sample_matches.iloc[0]

create_match_features(sample_match, completed_matches)

{'date': Timestamp('2024-01-01 00:00:00'),
 'home_team': 'China',
 'away_team': 'Hong Kong',
 'home_win_rate': 0.4,
 'away_win_rate': 0.2,
 'home_avg_goals_scored': np.float64(1.3),
 'away_avg_goals_scored': np.float64(1.8),
 'home_avg_goals_conceded': np.float64(1.0),
 'away_avg_goals_conceded': np.float64(1.4),
 'home_goal_diff': np.float64(0.30000000000000004),
 'away_goal_diff': np.float64(0.40000000000000013),
 'neutral': 1,
 'result': 'A'}

In [56]:
feature_rows = []

for idx, (_, match) in enumerate(sample_matches.iterrows()):

    if idx % 100 == 0:
        print(f"Processed {idx}")

    feature_rows.append(create_match_features(match, completed_matches)) 

Processed 0
Processed 100
Processed 200
Processed 300
Processed 400
Processed 500
Processed 600
Processed 700
Processed 800
Processed 900
Processed 1000
Processed 1100
Processed 1200
Processed 1300
Processed 1400
Processed 1500
Processed 1600
Processed 1700
Processed 1800
Processed 1900
Processed 2000
Processed 2100
Processed 2200
Processed 2300
Processed 2400
Processed 2500


In [57]:
feature_df = pd.DataFrame(feature_rows)

In [58]:
feature_df.head()

,date,home_team,away_team,home_win_rate,away_win_rate,home_avg_goals_scored,away_avg_goals_scored,home_avg_goals_conceded,away_avg_goals_conceded,home_goal_diff,away_goal_diff,neutral,result
0,2024-01-01,China,Hong Kong,0.4,0.2,1.3,1.8,1.0,1.4,0.3,0.4,1,A
1,2024-01-01,Japan,Thailand,0.7,0.3,3.5,1.3,0.9,2.2,2.6,-0.9,0,H
2,2024-01-02,Indonesia,Libya,0.4,0.4,2.1,1.2,1.3,1.3,0.8,-0.1,1,A
3,2024-01-04,Tajikistan,Hong Kong,0.2,0.3,1.3,1.9,1.1,1.4,0.2,0.5,1,H
4,2024-01-04,Saudi Arabia,Lebanon,0.2,0.4,1.3,1.3,1.7,0.9,-0.4,0.4,1,H


In [59]:
feature_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2540 entries, 0 to 2539
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   date                     2540 non-null   datetime64[ns]
 1   home_team                2540 non-null   object        
 2   away_team                2540 non-null   object        
 3   home_win_rate            2539 non-null   float64       
 4   away_win_rate            2540 non-null   float64       
 5   home_avg_goals_scored    2539 non-null   float64       
 6   away_avg_goals_scored    2540 non-null   float64       
 7   home_avg_goals_conceded  2539 non-null   float64       
 8   away_avg_goals_conceded  2540 non-null   float64       
 9   home_goal_diff           2539 non-null   float64       
 10  away_goal_diff           2540 non-null   float64       
 11  neutral                  2540 non-null   int64         
 12  result                   2540 non-

In [60]:
feature_df.isnull().sum()

date                       0
home_team                  0
away_team                  0
home_win_rate              1
away_win_rate              0
home_avg_goals_scored      1
away_avg_goals_scored      0
home_avg_goals_conceded    1
away_avg_goals_conceded    0
home_goal_diff             1
away_goal_diff             0
neutral                    0
result                     0
dtype: int64

In [61]:
feature_df[feature_df["home_goal_diff"].isna()].head(20)

,date,home_team,away_team,home_win_rate,away_win_rate,home_avg_goals_scored,away_avg_goals_scored,home_avg_goals_conceded,away_avg_goals_conceded,home_goal_diff,away_goal_diff,neutral,result
1661,2025-08-14,Marshall Islands,United States Virgin Islands,NaN,0.0,NaN,0.9,NaN,2.5,NaN,-1.6,1,A


In [62]:
model_df = feature_df.dropna().copy()

print(model_df.shape)

(2539, 13)


In [63]:
model_df.head()

,date,home_team,away_team,home_win_rate,away_win_rate,home_avg_goals_scored,away_avg_goals_scored,home_avg_goals_conceded,away_avg_goals_conceded,home_goal_diff,away_goal_diff,neutral,result
0,2024-01-01,China,Hong Kong,0.4,0.2,1.3,1.8,1.0,1.4,0.3,0.4,1,A
1,2024-01-01,Japan,Thailand,0.7,0.3,3.5,1.3,0.9,2.2,2.6,-0.9,0,H
2,2024-01-02,Indonesia,Libya,0.4,0.4,2.1,1.2,1.3,1.3,0.8,-0.1,1,A
3,2024-01-04,Tajikistan,Hong Kong,0.2,0.3,1.3,1.9,1.1,1.4,0.2,0.5,1,H
4,2024-01-04,Saudi Arabia,Lebanon,0.2,0.4,1.3,1.3,1.7,0.9,-0.4,0.4,1,H
